In [79]:
# 忽略各模块的警告信息
import warnings
warnings.filterwarnings("ignore")

In [100]:
# 加载所需使用的模块
from datetime import datetime

from vnpy.trader.datafeed import get_datafeed
from vnpy.trader.database import get_database, DB_TZ
from vnpy.trader.constant import Interval
from vnpy.trader.object import BarData, HistoryRequest
from vnpy.trader.utility import extract_vt_symbol
from vnpy.trader.setting import SETTINGS
from xtquant import xtdata

In [119]:
sector_name = '沪深A股'
code_list = xtdata.get_stock_list_in_sector(sector_name)
# print(f'{sector_name}一共有{len(code_list)}只品种')
# print('-------')
# print(code_list)
# xtdata.get_instrument_detail("600238.SH")

# xtdata.get_financial_data(["600238.SH"], table_list=["Balance"], start_time='20251011', end_time='20260415', report_type='report_time')

xtdata.download_financial_data(["600238.SH"], table_list=["Balance","Income","CashFlow"])

Exception: 行情服务连接断开

In [81]:
# 配置数据服务
SETTINGS["datafeed.name"] = "xt"            # 可以根据自己的需求选择数据服务：rqdata/xt/wind等
SETTINGS["datafeed.username"] = "client"       # RQData的用户名统一为“license”这个字符串
SETTINGS["datafeed.password"] = ""        # 这里需要替换为你购买或者申请试用的RQData数据license

# 配置数据库
SETTINGS["database.name"] = "postgresql"
SETTINGS["database.host"] = "localhost"
SETTINGS["database.port"] = "5432"
SETTINGS["database.database"] = "vnpy"
SETTINGS["database.user"] = "guojiantao"
SETTINGS["database.password"] = "123456"

In [82]:
# 创建对象实例
datafeed = get_datafeed()

database = get_database()

In [83]:
# 要下载数据的合约代码
vt_symbols = [
    "600988.SSE",
    "IF2502.CFFEX",
    "IF2503.CFFEX",

    "IH2501.CFFEX",
    "IH2502.CFFEX",
    "IH2503.CFFEX",

    "IC2501.CFFEX",
    "IC2502.CFFEX",
    "IC2503.CFFEX",

    "IM2501.CFFEX",
    "IM2502.CFFEX",
    "IM2503.CFFEX",
]

In [84]:
# 要下载数据的起止时间
start = datetime(2026, 1, 1, tzinfo=DB_TZ)
end = datetime(2026, 3, 30, tzinfo=DB_TZ)

In [85]:
# 遍历列表执行下载
for vt_symbol in vt_symbols:
    # 拆分合约代码和交易所
    symbol, exchange = extract_vt_symbol(vt_symbol)

    # 创建历史数据请求对象
    req: HistoryRequest = HistoryRequest(
        symbol=symbol,
        exchange=exchange,
        start=start,
        end=end,
        interval=Interval.DAILY        # 这里下载最常用的1分钟K线
    )

    # 从数据服务下载数据
    bars: list[BarData] = datafeed.query_bar_history(req)

    # 如果下载成功则保存
    if bars:
        database.save_bar_data(bars)
        print(f"下载数据成功：{vt_symbol}，总数据量：{len(bars)}")
    # 否则失败则打印信息
    else:
        print(f"下载数据失败：{vt_symbol}")

下载数据成功：600988.SSE，总数据量：54
下载数据失败：IF2502.CFFEX
下载数据失败：IF2503.CFFEX
下载数据失败：IH2501.CFFEX
下载数据失败：IH2502.CFFEX
下载数据失败：IH2503.CFFEX
下载数据失败：IC2501.CFFEX
下载数据失败：IC2502.CFFEX
下载数据失败：IC2503.CFFEX
下载数据失败：IM2501.CFFEX
下载数据失败：IM2502.CFFEX
下载数据失败：IM2503.CFFEX
